In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window as w

spark = SparkSession.builder.appName('ECOM').getOrCreate()

customers = spark.read.csv('/Volumes/mycatalog/myschema/ecom_db/customers.csv', header = True, inferSchema = True)
order_items = spark.read.csv('/Volumes/mycatalog/myschema/ecom_db/order_items.csv', header = True, inferSchema = True)
orders = spark.read.csv('/Volumes/mycatalog/myschema/ecom_db/orders.csv', header = True, inferSchema = True)
payments = spark.read.csv('/Volumes/mycatalog/myschema/ecom_db/payments.csv', header = True, inferSchema = True)
product_reviews = spark.read.csv('/Volumes/mycatalog/myschema/ecom_db/product_reviews.csv', header = True, inferSchema = True)
products = spark.read.csv('/Volumes/mycatalog/myschema/ecom_db/products.csv', header = True, inferSchema = True)

spark.sql('USE mycatalog.myschema')


In [0]:
# print(customers.rdd.getNumPartitions())
# print(order_items.rdd.getNumPartitions())
# print(orders.rdd.getNumPartitions())
# print(payments.rdd.getNumPartitions())
# print(product_reviews.rdd.getNumPartitions())
# print(products.rdd.getNumPartitions())


In [0]:
customers.write.format("delta").mode("overwrite").saveAsTable('mycatalog.myschema.customers')
order_items.write.format("delta").mode("overwrite").saveAsTable('mycatalog.myschema.order_items')
orders.write.format("delta").mode("overwrite").saveAsTable('mycatalog.myschema.orders')
payments.write.format("delta").mode("overwrite").saveAsTable('mycatalog.myschema.payments')
product_reviews.write.format("delta").mode("overwrite").saveAsTable('mycatalog.myschema.product_reviews')
products.write.format("delta").mode("overwrite").saveAsTable('mycatalog.myschema.products')

**1. Retrieve the top 10 customers who have spent the highest total amount across all orders in the past 12 months, considering only successfully paid orders.**

In [0]:
# select a.customer_id, a.name, b.total_amount
# from customers a
# left join orders b on b.customer_id = a.customer_id
# where b.payment_status = 'PAID' and timestampdiff(month, b.order_date, curdate()) < 12
# order by b.total_amount desc
# limit 10;

# customers.alias('a')\
#          .join(orders.alias('b'), col('a.customer_id') == col('b.customer_id'), 'left')\
#          .where('b.payment_status = "PAID" and timestampdiff(month, b.order_date, curdate()) < 12')\
#          .select('a.customer_id', 'a.name', 'b.total_amount', 'b.order_date', 'b.payment_status')\
#          .orderBy(col('b.total_amount').desc())\
#          .limit(10)\
#          .display()

view_1 = customers.alias('a') \
                  .join(orders.alias('b'), customers.customer_id == orders.customer_id, 'left') \
                  .where((orders.payment_status == 'PAID') & 
                         (orders.order_date >= date_sub(curdate(), 365)))\
                  .select('a.customer_id', 
                          'a.name', 
                          'b.total_amount', 
                          col('b.order_date').cast('string').alias('order_date'), 
                          'b.payment_status')\
                  .orderBy(col('b.total_amount').desc())\
                  .limit(10)

# spark.sql('drop table if exists insigth_1')
# view_1.write.format('delta').mode('overwrite').saveAsTable('insigth_1')

# test = spark.sql('select * from insigth_1 limit 5').collect()
test = view_1.collect()

for row in test:
        for column in row.asDict().keys():
                print(column)
        break

        


**2. Identify all customers who registered on the platform but have never placed a single order.**

In [0]:
# select a.customer_id, a.name, a.registration_datetime
# from customers a
# left join orders b on b.customer_id = a.customer_id
# where b.customer_id is null;

# customers.alias('a')\
#          .join(orders.alias('b'), col('a.customer_id') == col('b.customer_id'), 'left')\
#          .where('b.customer_id is NULL')\
#          .orderBy('a.customer_id')\
#          .display()


customers.join(orders, customers.customer_id == orders.customer_id, 'left')\
         .where(orders.customer_id.isNull())\
         .orderBy(customers.customer_id)\
         .display()

**3. Find customers who have placed more than 5 orders and have also submitted at least 2 product reviews.**

In [0]:
# with cte1 as (
# select customer_id, count(*) as orders_placed
# from orders
# group by customer_id
# having count(*) > 5
# order by customer_id
# ),

# cte2 as (
# select customer_id, count(*) as reviews_placed
# from product_reviews
# group by customer_id
# having count(*) >= 2
# order by customer_id
# )

# select a.customer_id, a.orders_placed, b.reviews_placed
# from cte1 a
# join cte2 b on b.customer_id = a.customer_id
# order by a.customer_id

# cte1 = orders.groupBy(orders.customer_id)\
#              .agg(count('*').alias('orders_placed'))\
#              .where('orders_placed > 5')

# cte2 = product_reviews.groupBy(product_reviews.customer_id)\
#                       .agg(count('*').alias('reviews_placed'))\
#                       .where('reviews_placed >= 2')

# cte1.join(cte2, cte1.customer_id == cte2.customer_id)\
#     .select(cte1.customer_id, cte1.orders_placed, cte2.reviews_placed)\
#     .orderBy(cte1.customer_id)\
#     .display()

orders.groupBy(orders.customer_id)\
      .agg(count('*').alias('orders_placed'))\
      .where('orders_placed > 5')\
      .join(product_reviews, product_reviews.customer_id == orders.customer_id)\
      .groupBy(orders.customer_id, 'orders_placed')\
      .agg(count('*').alias('Reviews_submitted'))\
      .where('Reviews_submitted >= 2')\
      .orderBy(orders.customer_id)\
      .display()

**4. Calculate the average customer lifetime value by summing the total amount of all successful orders per customer.**

In [0]:
# select customer_id, round(avg(total_amount), 2) as average_customer_lifetime_value
# from orders
# where payment_status = 'PAID'
# group by customer_id
# order by customer_id

orders.where(orders.payment_status == 'PAID')\
      .groupBy(orders.customer_id)\
      .agg(round(avg(orders.total_amount), 2).alias('average_customer_lifetime_value'))\
      .orderBy(orders.customer_id)\
      .display()

**5. List all customers who have purchased products from more than 3 distinct product categories.**

In [0]:
# select a.customer_id, count(distinct c.category) as distinct_product_categories
# from orders a
# join order_items b on b.order_id = a.order_id
# join products c on c.product_id = b.product_id
# group by a.customer_id
# having count(distinct c.category) >= 3
# order by 1;

orders.join(order_items, order_items.order_id == orders.order_id)\
      .join(products, products.product_id == order_items.product_id)\
      .groupBy(orders.customer_id)\
      .agg(countDistinct(products.category).alias('distinct_product_categories'))\
      .where('distinct_product_categories >= 3')\
      .orderBy(orders.customer_id)\
      .display()

**6. Identify customers who have placed at least one order but do not have any associated successful payment.**

In [0]:
# select distinct a.customer_id
# from orders a
# join payments b on b.order_id = a.order_id
# where not b.payment_successful
# order by a.customer_id;

orders.join(payments, payments.order_id == orders.order_id)\
      .where(~ payments.payment_successful)\
      .select(orders.customer_id).distinct()\
      .orderBy(orders.customer_id)\
      .display()

**7. Retrieve a list of customers who have not placed any orders in the last 12 months and are still marked as active.**

In [0]:
# with cte1 as (
# select customer_id, max(order_date) as latest_order_date
# from orders
# group by customer_id
# order by 1
# )

# select b.customer_id, b.latest_order_date, a.is_active
# from cte1 b
# join customers a on a.customer_id = b.customer_id
# where a.is_active and b.latest_order_date < subdate(curdate(), interval 365 day)
# order by 1;

orders.groupBy(orders.customer_id)\
      .agg(max(orders.order_date).alias('latest_order_date'))\
      .join(customers, customers.customer_id == orders.customer_id)\
      .where((col('latest_order_date') < date_sub(curdate(), 365)) & (customers.is_active))\
      .select(orders.customer_id, col('latest_order_date'), customers.is_active)\
      .orderBy(1)\
      .display()

**8. Find customers who submitted more than 3 product reviews where the rating was below 2.5 stars.**

In [0]:
# select customer_id, count(*) as product_count
# from product_reviews
# where rating_score < 2.5
# group by customer_id
# having count(*) > 3
# order by 1;

product_reviews.where(product_reviews.rating_score < 2.5)\
               .groupBy(product_reviews.customer_id)\
               .agg(count('*').alias('product_reviews'))\
               .where('product_reviews > 3')\
               .orderBy(1)\
               .display()


**9. Calculate the average time (in days) between a customer’s registration date and their first order date.**

In [0]:
# with cte1 as (
# select customer_id, min(order_date) as first_order_Date
# from orders
# group by customer_id
# ),

# cte2 as (
# select a.customer_id, date(a.registration_datetime) as registration_date, b.first_order_Date,
# 	   datediff(b.first_order_Date, date(a.registration_datetime)) as difference_in_days
# from customers a
# left join cte1 b on b.customer_id = a.customer_id
# where datediff(b.first_order_Date, date(a.registration_datetime)) > 0
# )

# select round(avg(difference_in_days)) as avg_difference_in_days
# from cte2

orders.groupBy(orders.customer_id)\
      .agg(min(orders.order_date).alias('first_order_date'))\
      .orderBy(1)\
      .join(customers, customers.customer_id == orders.customer_id, 'right')\
      .select(customers.customer_id, 
              'first_order_date', 
              to_date(customers.registration_datetime).alias('registration_date'),
              date_diff('first_order_date', to_date(customers.registration_datetime)).alias('diff'))\
      .where('diff > 0')\
      .groupBy()\
      .agg(round(avg('diff')).alias('avg_time'))\
      .display()



**10. Find all customers who have ordered the same product more than 5 times across all orders.**

In [0]:
# select customer_id, b.product_id, count(*) as order_count
# from orders a
# join order_items b on b.order_id = a.order_id
# group by customer_id, b.product_id
# having count(*) >= 5
# order by 1, 2

orders.join(order_items, order_items.order_id == orders.order_id)\
      .groupBy(orders.customer_id, order_items.product_id)\
      .agg(count('*').alias('order_count'))\
      .where('order_count >= 5')\
      .orderBy(1, 2)\
      .display()

**11. List the top 10 products that generated the highest total revenue from successful order items.**

In [0]:
# select b.product_id, round(sum(a.total_amount), 2) as total_revenue
# from orders a
# join order_items b on b.order_id = a.order_id
# where a.payment_status = 'PAID'
# group by b.product_id
# order by round(sum(a.total_amount), 2) desc
# limit 10;

orders.join(order_items, order_items.order_id == orders.order_id)\
      .where(orders.payment_status == 'PAID')\
      .groupBy(order_items.product_id)\
      .agg(round(sum(orders.total_amount), 2).alias('total_revenue'))\
      .orderBy(desc('total_revenue'))\
      .limit(10)\
      .display()

**12. Identify products that have been ordered more than 10 times but have never received any product reviews.**

In [0]:
# with cte1 as (
# select product_id, count(*) as product_count
# from order_items
# group by product_id
# having count(*) > 10
# )

# select a.product_id, a.product_count
# from cte1 a
# left join product_reviews b on b.product_id = a.product_id
# where b.product_id is null
# order by 1;


order_items.groupBy(order_items.product_id)\
           .agg(count('*').alias('product_count'))\
           .where('product_count > 10')\
           .join(product_reviews, product_reviews.product_id == order_items.product_id, 'left')\
           .where(product_reviews.product_id.isNull())\
           .select(order_items.product_id, 'product_count')\
           .orderBy(1)\
           .display()

**13.Find all products that currently have zero stock available but have been part of an order in the last 30 days.**

In [0]:
# select a.product_id, a.stock_quantity, max(c.order_date)
# from products a
# join order_items b on b.product_id = a.product_id
# join orders c on c.order_id = b.order_id
# where stock_quantity = 0
# group by a.product_id, a.stock_quantity
# having max(c.order_date) >= date_sub(curdate(), interval 30 day)

products.join(order_items, order_items.product_id == products.product_id)\
        .join(orders, orders.order_id == order_items.order_id)\
        .groupBy(products.product_id, products.stock_quantity)\
        .agg(max(orders.order_date).alias('latest_order_date'))\
        .where((products.stock_quantity == 0) & (col('latest_order_date') >= date_sub(curdate(), 30)))\
        .orderBy(1)\
        .display()

**14. List all products that have received only 5-star ratings in reviews and have at least 3 reviews.**

In [0]:
# select product_id, count(*) as product_count, sum(rating_score) as total_rating 
# from product_reviews
# group by product_id
# having count(*) >= 3 and sum(rating_score) = (count(*) * 5);

product_reviews.groupBy(product_reviews.product_id)\
               .agg(count('*').alias('product_count'), 
                    sum(product_reviews.rating_score).alias('total_rating'))\
               .where('product_count >= 3 and total_rating = (product_count * 5)')\
               .display()

**15. Calculate the average product rating per category using all submitted product reviews.**

In [0]:
# select a.category, avg(b.rating_score) as avg_rating
# from products a
# join product_reviews b on b.product_id = a.product_id
# group by a.category;

product_reviews.join(products, products.product_id == product_reviews.product_id)\
               .groupBy(products.category)\
               .agg(avg(product_reviews.rating_score).alias('avg_rating'))\
               .display()


**16. Identify products that were sold at more than 50% discount but still generated at least ₹50,000 in total revenue.**

In [0]:
# select a.product_id, a.discount_percent, sum(c.total_amount) as total_revenue
# from products a
# join order_items b on b.product_id = a.product_id
# join orders c on c.order_id = b.order_id
# where discount_percent > 50
# group by a.product_id, a.discount_percent
# having sum(c.total_amount) >= 50000;

products.join(order_items, order_items.product_id == products.product_id)\
        .join(orders, orders.order_id == order_items.order_id)\
        .groupBy(products.product_id, products.discount_percent)\
        .agg(round(sum(orders.total_amount)).alias('total_revenue'))\
        .where('discount_percent > 50 and total_revenue >= 50000')\
        .display()

**17. Retrieve all products that were ordered in the last 30 days where the order item quantity was greater than 3.**

In [0]:
# select a.product_id, a.item_id, a.quantity, b.order_date
# from order_items a
# join orders b on b.order_id = a.order_id
# where a.quantity > 3 and b.order_date >= date_sub(curdate(), interval 30 day)
# order by 1, 2

order_items.join(orders, orders.order_id == order_items.order_id)\
           .select(order_items.product_id,
                   order_items.item_id,
                   order_items.quantity,
                   orders.order_date)\
           .where('quantity > 3 and order_date >= date_sub(curdate(), 30)')\
           .orderBy(1, 2)\
           .display()

**18. Find the most reviewed product in each category based on the number of review entries.**

In [0]:
# with cte1 as (
# select b.category, b.product_id, b.name, count(*) as review_count,
# 	     dense_rank() over(partition by category order by count(*) desc) as rnk
# from product_reviews a
# join products b on b.product_id = a.product_id
# group by b.category, b.product_id, b.name
# order by 1, 2
# )

# select *
# from cte1
# where rnk = 1;

products.join(product_reviews, product_reviews.product_id == products.product_id)\
        .groupBy(products.category, products.product_id, products.name)\
        .agg(count('*').alias('total_reviews'))\
        .withColumn('rank', dense_rank()\
                            .over(w.partitionBy(products.category).orderBy(desc('total_reviews'))))\
        .where('rank = 1')\
        .display()

**19. Identify products that have less than 10 items in stock but appear in more than 100 orders in the past 60 days.**

In [0]:
# select b.product_id, count(*) as count
# from orders a
# join order_items b on b.order_id = a.order_id
# join products c on c.product_id = b.product_id
# where a.order_date >= date_sub(curdate(), interval 60 day) and c.stock_quantity < 10
# group by b.product_id
# having count(*) > 100
# order by 1;

orders.join(order_items, order_items.order_id == orders.order_id)\
      .join(products, products.product_id == order_items.product_id)\
      .where('order_Date >= date_sub(curdate(), 60)')\
      .groupBy(order_items.product_id, products.stock_quantity)\
      .agg(count('*').alias('count'))\
      .where('stock_quantity < 10 and count > 100')\
      .orderBy(1)\
      .display()

**20. List the top 5 products with the highest average length (in characters) of review text.**

In [0]:
# select product_id, round(avg(length(review_text)), 2) as avg_review_length
# from product_reviews
# group by product_id
# order by avg(length(review_text)) desc
# limit 5;

product_reviews.groupBy(product_reviews.product_id)\
               .agg(round(avg(length(product_reviews.review_text)), 2).alias('avg_review_length'))\
               .orderBy(desc('avg_review_length'))\
               .limit(5)\
               .display()

**21. Generate a report of monthly revenue for the past 12 months using only payments marked as successful.**

In [0]:
# select month(order_date) as sr, monthname(order_date) as month, sum(total_amount) as revenue
# from orders
# where payment_status = 'PAID' and order_date >= subdate(curdate(), interval 1 year)
# group by month(order_date), monthname(order_date)
# order by 1;

# orders.where('order_Date >= date_sub(curdate(), 365) and payment_status = "PAID"')\
#       .groupBy(month(orders.order_date).alias('Sr'), monthname(orders.order_date).alias('month'))\
#       .agg(round(sum(orders.total_amount),2).alias('revenue'))\
#       .orderBy(1)\
#       .display()



**22. Retrieve all failed payments and join with the customer details (name, email, phone) who attempted those payments.**

In [0]:
# select a.name, a.email, a.phone, b.order_id, b.payment_status
# from customers a
# join orders b on b.customer_id = a.customer_id
# where b.payment_status = 'FAILED'
# order by a.customer_id;


# customers.join(orders, orders.customer_id == customers. customer_id)\
#          .select(customers.name, 
#                  customers.email, 
#                  customers.phone, 
#                  orders.order_id, 
#                  orders.payment_status)\
#          .where(orders.payment_status == 'FAILED')\
#          .orderBy(customers.customer_id)\
#          .display()

orders.orderBy(orders.customer_id)
customers.orderBy(customers.customer_id)

df2 = customers.join(orders, orders.customer_id == customers. customer_id)\
         .select(customers.name, 
                 customers.email, 
                 customers.phone, 
                 orders.order_id, 
                 orders.payment_status)\
         .where(orders.payment_status == 'FAILED')\
         .orderBy(customers.customer_id)

df2.explain(True)


**program to fetch nth highest total_amount from Orders**

In [0]:
n = 1
temp = orders.collect()
max_amount = 1 + orders.select(max('total_amount')).collect()[0][0]

if n > 1:
    max_amount = orders.select('total_amount')\
                    .orderBy(desc('total_amount'))\
                    .collect()[n-2][0]

orders.where(orders.total_amount < max_amount)\
      .select("total_amount")\
      .orderBy(desc('total_amount'))\
      .limit(1)\
      .display()

# orders.select('total_amount')\
#       .orderBy(desc('total_amount'))\
#       .display()

orders.select('total_amount',
              dense_rank().over(w.orderBy(desc('total_amount'))).alias('rank'))\
      .createOrReplaceTempView('total_amount_desc')




**23. Identify orders that were placed but never had a matching successful payment associated with them.**

In [0]:
orders.where(orders.payment_status != 'PAID')\
      .select(orders.order_id, orders.payment_status)\
      .display()

**24. Calculate the average order value per customer, based only on orders with successful payments.**

In [0]:
orders.where(orders.payment_status == 'PAID')\
      .groupBy(orders.customer_id)\
      .agg(round(avg(orders.total_amount), 2).alias('avg_order_value'))\
      .orderBy(orders.customer_id)\
      .display()

**25. Find all orders where the total order amount differs from the payment amount by more than 10%.**

In [0]:
orders.join(payments, payments.order_id == orders.order_id)\
      .select(orders.order_id, orders.total_amount, payments.amount)\
      .withColumn('diff', round(abs(orders.total_amount - payments.amount)/payments.amount*100, 2))\
      .where(col('diff') > 10)\
      .orderBy(orders.order_id)\
      .display()

In [0]:
# null_check = customers.select([sum(col(c).isNull().cast('int')).alias(c) for c in customers.columns]).display()

# str_columns = [column for column in customers.columns if isinstance(customers.schema[column].dataType, StringType)]

str_columns = [column for column in customers.columns if customers.schema[column].dataType == StringType()]

# print(str_columns)

# customers.select(str_columns).display()

# empty_check = customers.select([sum((col(column) == '').cast('int')).alias(column) for column in str_columns]).display()

customers.select([sum((col(column) == '').cast('int')).alias(column) for column in customers.columns if customers.schema[column].dataType == StringType()])\
         .display()

